In [1]:
using Glob 
using DelimitedFiles, DataFrames, EzXML
using Dates
using LinearAlgebra
using ForwardDiff
using Statistics
using Interpolations
using Rotations
using Unitful
using ThreadsX
using WENO4
using Polyester
using CoordinateTransformations
import PhysicalConstants.CODATA2018: m_p, m_e, e, ε_0, μ_0, c_0, G, k_B
const M_Mars = 0.64169e24u"kg"
const R_Mars = 3389.5u"km"  #m
const au = 149597870.7u"km"

include("../../src/scripts/MAVEN_load.jl")
include("../../src/scripts/MAVEN_STATIC.jl")
include("../../src/scripts/SphericalSplines.jl")
using .MAVEN_STATIC
using .MAVEN_load
using .SphericalSplines

In [2]:
date = Date(2022, 8, 24)
datestr = Dates.format.(date, "yyyymmdd")
year_num = Dates.format(date, "yyyy")
month_num = Dates.format(date, "mm")

ts = DateTime(date) + Dates.Hour(7) + Dates.Minute(26)
te = ts + Dates.Minute(1)

2022-08-24T07:27:00

In [3]:
data_dir = "E:\\Data"
output_dir = "E:\\Output"
MAVEN_path = joinpath(data_dir, "MAVEN")
MAG_path = joinpath(MAVEN_path, "MAG", "l2", year_num, month_num)
STATIC_path = joinpath(MAVEN_path, "STATIC", "l2", year_num, month_num)
MAVEN_output_path = joinpath(output_dir, "MAVEN")
STATIC_output_path = joinpath(MAVEN_output_path, "STATIC")
VDF_path = joinpath(STATIC_output_path, "VDF", year_num, month_num)
mkpath(VDF_path)
MAG_file = glob("mvn_mag_l2*$datestr*.sts", MAG_path)
STA_file = glob("mvn_sta_l2*d1*32e4d16a8m*$datestr*.cdf", STATIC_path)
println("MAG file: ", MAG_file)
println("STATIC file: ", STA_file)
println("Output path: ", VDF_path)

MVN_mag2c1hz = MAVEN_load.load_mag_l2(MAG_file[1])
MVN_static = MAVEN_load.load_cdf(STA_file[1])
println("Data loaded.")

MAG file: ["E:\\Data\\MAVEN\\MAG\\l2\\2022\\08\\mvn_mag_l2_2022236ss_20220824_v01_r01.sts"]
STATIC file: ["E:\\Data\\MAVEN\\STATIC\\l2\\2022\\08\\mvn_sta_l2_d1-32e4d16a8m_20220824_v02_r03.cdf"]
Output path: E:\Output\MAVEN\STATIC\VDF\2022\08
Data loaded.


In [25]:
MVN_mag2c1hz

Dict{Symbol, Any} with 6 entries:
  :B_total   => Float32[8.44454, 9.03155, 9.08005, 9.4868, 9.71352, 9.07216, 7.…
  :JulUTtime => [2.45982e6, 2.45982e6, 2.45982e6, 2.45982e6, 2.45982e6, 2.45982…
  :type      => "36ss"
  :B         => Float32[6.62 -0.87 -5.17; 6.42 -1.29 -6.22; … ; -2.39 -2.53 -2.…
  :position  => Float32[2303.23 -1533.46 5824.94; 2303.3 -1533.47 5824.94; … ; …
  :epoch     => [DateTime("2022-08-24T00:00:01.105"), DateTime("2022-08-24T00:0…

In [34]:
# MVN MAG
datejul = datetime2julian(DateTime(date))
MVN_magut1hz = MVN_mag2c1hz[:JulUTtime].-datejul
MVN_BMSO1hz = MVN_mag2c1hz[:B]
ind = ThreadsX.findall(ts .<= MVN_mag2c1hz[:epoch] .<= te)
MVN_Bmeanmso = dropdims(mean(Matrix(MVN_BMSO1hz[ind, :]), dims=(1,)), dims=(1,))

3-element Vector{Float32}:
  1.6145784
 -5.4549227
 -1.6676201

In [44]:
# MVN STATIC
MVN_staticswp = MVN_static[:swp_ind] |> Array 
MVN_staticswp .+= 1
MVN_statictime = MVN_static[:epoch] |> Array
MVN_staticjulian=datetime2julian.(DateTime.(MVN_statictime))
MVN_staticeflux = MVN_static[:eflux] |> Array
MVN_staticmass = MVN_static[:mass_arr] |> Array
MVN_staticenergy = MVN_static[:energy] |> Array
MVN_statictheta = MVN_static[:theta] |> Array #-45 to 45 deg, corresponding to latitude in the STATIC/APP coordinate system
MVN_staticphi = MVN_static[:phi] |> Array #-180 to 180 deg, corresponding to longitude in the STATIC/APP coordinate system
MVN_staticdenergy = MVN_static[:denergy] |> Array
MVN_staticdtheta = MVN_static[:dtheta] |> Array
MVN_staticdphi = MVN_static[:dphi] |> Array
MVN_staticdomega = MVN_static[:domega] |> Array
MVN_staticvelocity = sqrt.(MVN_staticenergy.*(2.0/ustrip(u"eV", m_p*c_0^2))./MVN_staticmass).*ustrip(u"km/s", c_0)
MVN_staticpos_sc_mso = MVN_static[:pos_sc_mso] |> Array
MVN_staticpos_sc_mso /= ustrip(u"km", R_Mars)

21600×3 Matrix{Float64}:
 0.682265  -0.452666    1.71842
 0.685074  -0.453091    1.71831
 0.687881  -0.453515    1.7182
 0.690687  -0.453939    1.71808
 0.69349   -0.454361    1.71796
 0.696292  -0.454782    1.71783
 0.699092  -0.455201    1.7177
 0.701891  -0.45562     1.71756
 0.704687  -0.456037    1.71742
 0.707482  -0.456454    1.71728
 0.710275  -0.456869    1.71713
 0.713067  -0.457283    1.71697
 0.715856  -0.457696    1.71682
 ⋮                     
 1.23145    0.0315626  -1.23966
 1.22864    0.0323103  -1.24079
 1.22583    0.0330579  -1.24191
 1.22301    0.0338053  -1.24304
 1.22019    0.0345527  -1.24415
 1.21736    0.0352999  -1.24527
 1.21453    0.0360471  -1.24637
 1.21169    0.0367941  -1.24748
 1.20885    0.037541   -1.24858
 1.20601    0.0382878  -1.24967
 1.20316    0.0390344  -1.25077
 1.20031    0.039781   -1.25185

In [45]:
MVN_staticquat_mso = MVN_static[:quat_mso] |> Array #Quaternion elements to rotate from STATIC coordinates (same as APP coordinates) to MSO coordinates (NUM_DISTS, 3)
MVN_staticquat_sc = MVN_static[:quat_sc] |> Array #Quaternion elements to rotate from STATIC coordinates (same as APP coordinates) to SC coordinates (NUM_DISTS, 3)   
MVN_staticmsotrans = [QuatRotation(quat) for quat in eachcol(MVN_staticquat_mso')] # active (right handed) rotations where applicable.
MVN_staticind = ThreadsX.findall(ts .<= MVN_statictime .<= te)
@show minimum(MVN_staticswp[MVN_staticind]), maximum(MVN_staticswp[MVN_staticind])
MVN_staticswpind = MVN_staticswp[MVN_staticind[1]]
MVN_staticefluxmean = dropdims(mean(MVN_staticeflux[MVN_staticind, :, :, :], dims = (1, ) ), dims=(1, ))
MVN_staticpsdmean = ustrip(u"cm^-3 * km^-3 * s^3", 0.5 * 1u"cm^-2 * s^-1" * (m_p/1u"eV")^2)  .* MVN_staticefluxmean .* (MVN_staticmass[:,:,:,MVN_staticswpind,1]./ MVN_staticenergy[:,:,:,MVN_staticswpind,1]).^2
size(MVN_staticefluxmean)
size(MVN_staticmass[:,:,:,MVN_staticswpind,1])
size(MVN_staticenergy[:,:,:,MVN_staticswpind,1])
MVN_staticefluxmid = similar(MVN_staticefluxmean)
MVN_staticefluxuquant = similar(MVN_staticefluxmean)
MVN_staticefluxlquant = similar(MVN_staticefluxmean)
for i in axes(MVN_staticeflux, 2), j in axes(MVN_staticeflux, 3), k in axes(MVN_staticeflux, 4)
    data = MVN_staticeflux[MVN_staticind, i, j, k]
    MVN_staticefluxmid[i, j, k] = quantile(data, 0.5)
    MVN_staticefluxuquant[i, j, k] = quantile(data, 0.75)
    MVN_staticefluxlquant[i, j, k] = quantile(data, 0.25)
end
i, j =2, 14
MVN_staticenergyl = zeros(32)
MVN_staticenergyu = zeros(32)
for ie in 1:32
    R = MVN_staticdenergy[1, (j-1)*4+i, ie, MVN_staticswpind, 1]./MVN_staticenergy[1, (j-1)*4+i, ie, MVN_staticswpind, 1]
    ratio = 0.5R+sqrt(1.0+0.25R^2)
    MVN_staticenergyl[ie] = MVN_staticenergy[1, (j-1)*4+i, ie, MVN_staticswpind, 1]/ratio
    MVN_staticenergyu[ie] = MVN_staticenergy[1, (j-1)*4+i, ie, MVN_staticswpind, 1]*ratio
end
MVN_staticenergygridH = vcat(MVN_staticenergyu, MVN_staticenergyl[end])
MVN_staticvelocitygridH = reverse(sqrt.(MVN_staticenergygridH.*(2.0/ustrip(u"eV", m_p*c_0^2))/1.03).*ustrip(u"km/s", c_0))
MVN_staticphigridH = [MVN_staticphi[1, (j-1)*4+1, 7, MVN_staticswpind, 1]-MVN_staticdphi[1, (j-1)*4+1, 7, MVN_staticswpind, 1]*0.5 for j in 1:16]
MVN_staticphigridH = vcat(MVN_staticphigridH, MVN_staticphigridH[begin]+360.0)
MVN_staticthetau = 90.0.-[MVN_statictheta[1, i, 7, MVN_staticswpind, 1]-MVN_staticdtheta[1, i, 7, MVN_staticswpind, 1]*0.5 for i in 1:4]
MVN_staticthetal = 90.0.-[MVN_statictheta[1, i, 7, MVN_staticswpind, 1]+MVN_staticdtheta[1, i, 7, MVN_staticswpind, 1]*0.5 for i in 1:4]
MVN_staticthetagridH = vcat(MVN_staticthetal, MVN_staticthetau[end])
MVN_staticindmean = round(Int64, mean(MVN_staticind))
MVN_statictimemean = MVN_statictime[MVN_staticindmean]
MVN_staticjulianmean = MVN_staticjulian[MVN_staticindmean]
MVN_static2msomean = QuatRotation(MVN_staticquat_mso[MVN_staticindmean, :]) # STATIC instrument to MSO transformation matrix at mean time
MVN_Bmeanstatic = MVN_static2msomean' * MVN_Bmeanmso #magnetic field in STATIC coordinates at mean time
MVN_parastatic = MVN_Bmeanstatic ./ norm(MVN_Bmeanstatic) #unit vector of magnetic field in STATIC coordinates at mean time
i, j =2, 14
ind = argmax(MVN_staticefluxmean[1, (j-1)*4+i, :] ./ MVN_staticenergy[1, 1, :, MVN_staticswpind, 1].^2)

(minimum(MVN_staticswp[MVN_staticind]), maximum(MVN_staticswp[MVN_staticind])) = (25, 25)


25

In [46]:
MVN_staticvswrough = CartesianFromSpherical()(CoordinateTransformations.Spherical(MVN_staticvelocity[1, (j-1)*4+i, ind, MVN_staticswpind, 1], deg2rad(MVN_staticphi[1, (j-1)*4+i, ind, MVN_staticswpind, 1]), deg2rad(MVN_statictheta[1, (j-1)*4+i, ind, MVN_staticswpind, 1])))
MVN_staticdirections = [CartesianFromSpherical()(CoordinateTransformations.Spherical(1.0, deg2rad(MVN_staticphi[1, ij, ind, MVN_staticswpind, 1]), deg2rad(MVN_statictheta[1, ij, ind, MVN_staticswpind, 1]))) for ij in 1:64] #direction unit vectors
MVN_staticss = [] #interpolation splines
Snan = SphericalSplines.InterpolationSpline(stack(MVN_staticdirections; dims=1), fill(-30.0, 64))
for k in 1:32
    ind = findall(MVN_staticefluxmean[1, :, k] .> 0.0)
    if !isempty(ind)
        S = SphericalSplines.InterpolationSpline(stack(MVN_staticdirections[ind]; dims=1), log10.(MVN_staticefluxmean[1, ind, k]))
        push!(MVN_staticss, S)
    else
        push!(MVN_staticss, Snan)
    end  
end
MVN_perpstatic1 = cross(MVN_parastatic, MVN_staticvswrough) #perpendicular direction1: E=B×Vsw
MVN_perpstatic1 = MVN_perpstatic1 ./ norm(MVN_perpstatic1)
MVN_perpstatic2 = cross(MVN_perpstatic1, MVN_parastatic) #perpendicular direction 2: N=E×B
MVN_static2mfa = transpose(hcat(MVN_perpstatic1, MVN_parastatic, MVN_perpstatic2))

3×3 StaticArraysCore.SMatrix{3, 3, Float64, 9} with indices SOneTo(3)×SOneTo(3):
 -0.00710329   0.26917   -0.963067
 -0.961402    -0.26675   -0.0674636
 -0.275057     0.925415   0.260675

In [48]:
function MVN_STATIC_eflux_slice(p0, d)
    v = Vector{Float64}()
    v3 = Vector{Vector{Float64}}()
    eflux = Vector{Float64}()
    for k in 6:32 # energy index
        roots = line_sphere_intersection(p0, d, [0.0, 0.0, 0.0], MVN_staticvelocity[1, 1, k, MVN_staticswpind, 1])
        if !isnothing(roots)
            for root in roots
                push!(v, dot(root, d))
                push!(v3, root)
                sph = SphericalFromCartesian()(root)
                lat = rad2deg(sph.ϕ) 
                if abs(lat) <= 45.0
                    S = MVN_staticss[k]
                    new_direction = Vector(root ./ norm(root))
                    eflux_val = 10.0^S(new_direction)
                    push!(eflux, eflux_val)                  
                else
                    push!(eflux, NaN64)
                end    
            end
        end
    end
    ind = sortperm(v)
    v = v[ind]
    v3 = v3[ind]
    eflux = eflux[ind]
    
    return v, v3, eflux
end

function MVN_STATIC_eflux_npslice(p0, d)
    v = Vector{Float64}()
    v3 = Vector{Vector{Float64}}()
    eflux = Vector{Float64}()
    for k in 6:32 # energy index
        roots = line_sphere_intersection(p0, d, [0.0, 0.0, 0.0], MVN_staticvelocity[1, 1, k, MVN_staticswpind, 1])
        if !isnothing(roots)
            for root in roots
                push!(v, dot(root, d))
                push!(v3, root)
                sph = SphericalFromCartesian()(root)
                lat = rad2deg(sph.ϕ) 
                if abs(lat) <= 45.0
                    ind = argmin([angle(Array(x), Array(root)) for x in Array(MVN_staticdirections)])  #find the closest direction
                    push!(eflux, MVN_staticefluxmean[1, ind, k])                   
                else
                    push!(eflux, NaN64)
                end    
            end
        end
    end
    ind = sortperm(v)
    v = v[ind]
    v3 = v3[ind]
    eflux = eflux[ind]
    
    return v, v3, eflux
end

function MVN_STATIC_eflux_interp(v3)
    vs = SphericalFromCartesian()(v3) 
    vtot = vs.r
    lon = vs.θ
    lat = vs.ϕ
    if (abs(rad2deg(lat)) <= 45.0) & (MVN_staticvelocity[1, 1, 6, MVN_staticswpind, 1] ≥ vtot ≥ MVN_staticvelocity[1, 1, 32, MVN_staticswpind, 1])
        dir = v3 ./ norm(vtot)
        logv = log10.(MVN_staticvelocity[1, 1, 6:32, MVN_staticswpind, 1])
        logeflux = Vector{Float64}(undef, 27)
        for k in 6:32 # energy index
            S = MVN_staticss[k]
            logeflux[k-5] = S(dir)
        end
        x = reverse(logv)
        y = reverse(logeflux)
        return 10.0^interpolate_weno4([log10(vtot)], x, y)[1]
    else
        return NaN64
    end
end

function MVN_STATIC_eflux_npinterp(v3)
    vs = SphericalFromCartesian()(v3) 
    vtot = vs.r
    lon = vs.θ
    lat = vs.ϕ
    vtot = norm(v3)
    staticlon = MVN_staticphigridH  #-180 to 180
    staticlat = 90.0 .- MVN_staticthetagridH #45 to -45
    staticv = reverse(MVN_staticvelocitygridH) # from high to low
    if (abs(rad2deg(lat)) <= 45.866) & (MVN_staticvelocity[1, 1, 7, MVN_staticswpind, 1] ≥ vtot ≥ MVN_staticvelocity[1, 1, 32, MVN_staticswpind, 1])
        indlon = findfirst(x -> x ≥ rad2deg(lon), staticlon) - 1
        if indlon == 0
            indlon = 16
        end
        indlat = findfirst(x -> x ≤ rad2deg(lat), staticlat) -1
        indv = findfirst(x -> x ≤ vtot, staticv) -1

        #@show lon, lat, vtot, indlon, indlat
        return MVN_staticefluxmean[1, (indlon-1)*4+indlat, indv]
    else
        return NaN64
    end
end

MVN_STATIC_eflux_npinterp (generic function with 1 method)

In [70]:
vpara = collect(range(-1000.0, 1000.0, step = 10.0))
vperp1 = collect(range(-1000.0, 1000.0, step = 10.0))
vperp2 = collect(range(-1000.0, 1000.0, step = 10.0))
MVN_static_eflux_rect = Array{Float64, 3}(undef, length(vperp1), length(vpara), length(vperp2))
@batch for k in 1:length(vperp2)
    for j in 1:length(vpara)
        for i in 1:length(vperp1)
            v3 = Array(vpara[j] .* MVN_parastatic .+ vperp1[i] .* MVN_perpstatic1 .+ vperp2[k] .* MVN_perpstatic2)
            MVN_static_eflux_rect[i, j, k] = MVN_STATIC_eflux_interp(v3)
            #MVN_static_eflux_rect[i, j, k] = MVN_STATIC_eflux_npinterp(v3)
        end
    end
end

In [72]:
using GLMakie
GLMakie.activate!()

size_inches = (20, 9)
size_pt = 72 .* size_inches
fig = Figure(size = size_pt, fontsize = 15, figure_padding = (5, 10, 9, 8))
title_str = Dates.format(ts, "yyyymmddTHH:MM") * "-" * Dates.format(te, "HH:MM")
fig_name = "MAVEN_STATIC_H+_VDF_3D_" * Dates.format(ts, "yyyymmddTHHMMSS") *
    "-" * Dates.format(te, "HHMMSS") * ".png"
Label(fig[1, 1, Top()], title_str, fontsize = 18, tellheight = true)
ls1 = LScene(fig[1, 1])

hm_ref = heatmap!(ls1,
    vperp1, vperp2,
    MVN_static_eflux_rect[:, 1, :],   # 用第1个 vpara 切片
    transformation = (:xz, vpara[1]),
    colormap = :gist_earth,
    colorrange = (1e4, 1e8),
    colorscale = log10)

for ipara in 1:length(vpara)
    vparai = vpara[ipara]
    hm = heatmap!(ls1, vperp1, vperp2, MVN_static_eflux_rect[:, ipara, :], transformation = (:xz, vparai), 
        colormap = :gist_earth,  colorrange = (1e4, 1e8), colorscale = log10)
    if ipara == 1
        hm_ref = hm
    end
end
i, j = 1, 13
ind = argmax(MVN_staticefluxmean[1, j, i, :])
v1 = CartesianFromSpherical()(CoordinateTransformations.Spherical(MVN_staticvelocity[1, (j-1)*4+i, ind, MVN_staticswpind, 1], 
    deg2rad(MVN_staticphi[1, (j-1)*4+i, ind, MVN_staticswpind, 1]),
    deg2rad(MVN_statictheta[1, (j-1)*4+i, ind, MVN_staticswpind, 1])))
# v1 = CartesianFromSpherical()(CoordinateTransformations.Spherical(MVN_staticvelocity[1, (j-1)*4+i, 22, MVN_staticswpind, 1], 
#     deg2rad(MVN_staticphi[1, (j-1)*4+i, ind, MVN_staticswpind, 1]),
#     deg2rad(MVN_statictheta[1, (j-1)*4+i, ind, MVN_staticswpind, 1]+22.5)))
ps = Point3f(0.0, 0.0, 0.0)
pe = Point3f(dot(v1, MVN_perpstatic1), dot(v1, MVN_parastatic), dot(v1, MVN_perpstatic2))
arrows3d!(ls1, ps, pe, argmode = :endpoint, color = :red, shaftradius = 0.02, tipradius = 0.05, tailradius = 0.02)

Label(fig[1, 2, Top()], title_str, fontsize = 18, tellheight = true)
ls2 = LScene(fig[1, 2])

for iperp1 in 81:1:81
    vperp1i = vperp1[iperp1]
    # heatmap!(ls2, vpara, vperp2, TW_minpa_eflux_rect[iperp1, :, :], transformation = (:yz, vperp1i), colormap = :gist_earth,  colorrange = (1e4, 1e8), colorscale = log10)
    heatmap!(ls2, vpara, vperp2, MVN_static_eflux_rect[iperp1, :, :], transformation = (:yz, vperp1i), colormap = :gist_earth,  colorrange = (1e4, 1e8), colorscale = log10)
end

i, j = 1, 13
ind = argmax(MVN_staticefluxmean[1, j, i, :])
v1 = CartesianFromSpherical()(CoordinateTransformations.Spherical(MVN_staticvelocity[1, (j-1)*4+i, ind, MVN_staticswpind, 1], 
    deg2rad(MVN_staticphi[1, (j-1)*4+i, ind, MVN_staticswpind, 1]),
    deg2rad(MVN_statictheta[1, (j-1)*4+i, ind, MVN_staticswpind, 1])))
# v1 = CartesianFromSpherical()(CoordinateTransformations.Spherical(MVN_staticvelocity[1, (j-1)*4+i, 22, MVN_staticswpind, 1], 
#     deg2rad(MVN_staticphi[1, (j-1)*4+i, ind, MVN_staticswpind, 1]),
#     deg2rad(MVN_statictheta[1, (j-1)*4+i, ind, MVN_staticswpind, 1]+22.5)))
ps = Point3f(0.0, 0.0, 0.0)
pe = Point3f(dot(v1, MVN_perpstatic1), dot(v1, MVN_parastatic), dot(v1, MVN_perpstatic2))
arrows3d!(ls2, ps, pe, argmode = :endpoint, color = :red, shaftradius = 0.02, tipradius = 0.05, tailradius = 0.02)

Colorbar(fig[1, 3], hm_ref, label = "Differential Energy Flux")

display(fig)
# 持续显示窗口，直到用户关闭
while isopen(fig.scene)
    sleep(0.1)
end

In [ ]:
v1

In [63]:
MVN_staticvelocity[1, (j-1)*4+i, ind, MVN_staticswpind, 1]

2248.612003679844

In [58]:
MVN_staticvelocity

8×64×32×28 Array{Float64, 4}:
[:, :, 1, 1] =
 137.064    137.064    137.064    …  137.064    137.064    137.064
  62.5488    62.5488    62.5488       62.5488    62.5488    62.5488
  40.6062    40.6062    40.6062       40.6062    40.6062    40.6062
  30.2477    30.2477    30.2477       30.2477    30.2477    30.2477
  22.0122    22.0122    22.0122       22.0122    22.0122    22.0122
  16.4271    16.4271    16.4271   …   16.4271    16.4271    16.4271
  12.1028    12.1028    12.1028       12.1028    12.1028    12.1028
   9.22089    9.22089    9.22089       9.22089    9.22089    9.22089

[:, :, 2, 1] =
 128.125    128.125    128.125    …  128.125    128.125    128.125
  58.4695    58.4695    58.4695       58.4695    58.4695    58.4695
  37.9581    37.9581    37.9581       37.9581    37.9581    37.9581
  28.2752    28.2752    28.2752       28.2752    28.2752    28.2752
  20.5768    20.5768    20.5768       20.5768    20.5768    20.5768
  15.356     15.356     15.356    …   15.356     15.356 

In [57]:
i, j = 1, 13
ind = argmax(MVN_staticefluxmean[1, j, i, :])

1